# Olist — Diagnóstico Estratégico 2017–2018
**POSTECH — Tech Challenge DTAT | Fase 1**

Notebook simplificado com as análises essenciais da apresentação.

## 1. Setup e Carga dos Dados

In [1]:
!pip install kaleido==0.2.1 -q

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import kagglehub

# Download dataset
path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print('Path:', path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 13.0 MB/s eta 0:00:00
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path: /kaggle/input/brazilian-ecommerce


In [2]:
# Carregar tabelas
df_customers = pd.read_csv(path + '/olist_customers_dataset.csv')
df_order_items = pd.read_csv(path + '/olist_order_items_dataset.csv')
df_order_payments = pd.read_csv(path + '/olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv(path + '/olist_order_reviews_dataset.csv')
df_orders = pd.read_csv(path + '/olist_orders_dataset.csv')
df_products = pd.read_csv(path + '/olist_products_dataset.csv')
df_seller = pd.read_csv(path + '/olist_sellers_dataset.csv')

## 2. Preparação dos Dados

In [3]:
# Timestamps e filtros
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['month'] = df_orders['order_purchase_timestamp'].dt.month
df_orders['year'] = df_orders['order_purchase_timestamp'].dt.year
df_orders['month/year'] = df_orders['month'].astype(str) + '/' + df_orders['year'].astype(str)
df_orders.sort_values('order_purchase_timestamp', inplace=True)

# Apenas pedidos entregues
df_orders_delivered = df_orders[df_orders['order_status'] == 'delivered'].copy()

In [4]:
# Merge principal: orders + payments + customers + reviews
df_order_x_payments = pd.merge(df_order_payments, df_orders_delivered, on='order_id', how='inner')
df_customers_x_pay = pd.merge(df_order_x_payments, df_customers, on='customer_id', how='left')
df_customers_review = pd.merge(df_customers_x_pay, df_order_reviews, on='order_id', how='left')

# Atraso em dias
df_customers_review['atraso_dias'] = (
    pd.to_datetime(df_customers_review['order_delivered_customer_date']) -
    pd.to_datetime(df_customers_review['order_estimated_delivery_date'])
).dt.days

# Merge para categorias
df_order_item_x_pay = pd.merge(df_order_x_payments, df_order_items, on='order_id', how='left')
df_revenue_by_category = pd.merge(df_order_item_x_pay, df_products, on='product_id', how='left')

print(f'Pedidos entregues: {df_orders_delivered.shape[0]:,}')
print(f'Clientes únicos: {df_customers_review["customer_unique_id"].nunique():,}')

Pedidos entregues: 96,478
Clientes únicos: 93,357


## 3. Slide 2 — Receita Mensal (Crescimento → Estabilização)

In [16]:
df_customers_review['order_purchase_timestamp'] = pd.to_datetime(df_customers_review['order_purchase_timestamp'])
df_customers_review['ano_mes'] = df_customers_review['order_purchase_timestamp'].dt.to_period('M').astype(str)

df_receita_mes = df_customers_review.groupby('ano_mes')['payment_value'].sum().reset_index()
df_receita_mes = df_receita_mes[df_receita_mes['ano_mes'] >= '2017-01'].sort_values('ano_mes')

# Lista ordenada de meses para calcular posição relativa no papel (0 a 1)
meses = df_receita_mes['ano_mes'].tolist()
idx_nov2017 = meses.index('2017-11') if '2017-11' in meses else None

fig = go.Figure(go.Bar(
    x=df_receita_mes['ano_mes'],
    y=df_receita_mes['payment_value'],
    marker_color='#2980b9',
    text=[f'R$ {v/1e6:.1f}M' for v in df_receita_mes['payment_value']],
    textposition='outside',
    textfont=dict(size=10, color='#2c3e50', family='Arial')
))

# Linha divisória: usar o valor categórico real, não índice numérico
if idx_nov2017 is not None:
    fig.add_vline(
        x='2017-11',
        line=dict(color='red', width=2, dash='dash')
    )

# Anotações: usar xref='paper' (0 a 1) para não depender do eixo categórico
fig.add_annotation(
    x=0.25, y=1.05,          # 25% da largura do gráfico
    xref='paper', yref='paper',
    text='Crescimento Acelerado',
    showarrow=False,
    font=dict(size=12, color='green')
)
fig.add_annotation(
    x=0.75, y=1.05,          # 75% da largura do gráfico
    xref='paper', yref='paper',
    text='Estabilização',
    showarrow=False,
    font=dict(size=12, color='red')
)

fig.update_layout(
    title='Forte crescimento em 2017 — o que nos impede de crescer em 2018?',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=False, tickangle=-45),
    yaxis=dict(showgrid=False, showticklabels=False),
    width=1200, height=500,
    margin=dict(t=80, b=80, l=40, r=40)
)

# Exportar imagem — garantir kaleido instalado
try:
    fig.write_image('grafico_receita_mensal.png', scale=3)
except Exception as e:
    print(f"write_image falhou: {e}")
    print("Tente: pip install kaleido")

fig.show()

## 4. Slide 3 — Crescimento Relativo (Base 100)

In [6]:
# Sellers por mês
df_seller_x_pay = pd.merge(pd.merge(df_order_x_payments, df_order_items, on='order_id', how='left'), df_seller, on='seller_id', how='left')
df_seller_mes = df_seller_x_pay.groupby(['month/year','month','year'])['seller_id'].nunique().reset_index()
df_seller_mes.sort_values(['year','month'], inplace=True)
df_seller_mes = df_seller_mes[(df_seller_mes['year'] >= 2017)]

# Clientes e pedidos por mês
df_cli_mes = df_customers_x_pay.groupby(['month/year','month','year']).agg(
    clientes=('customer_unique_id', 'nunique'),
    pedidos=('order_id', 'nunique'),
    ticket=('payment_value', 'mean')
).reset_index()
df_cli_mes.sort_values(['year','month'], inplace=True)
df_cli_mes = df_cli_mes[df_cli_mes['year'] >= 2017]

# Base 100
base_s = df_seller_mes['seller_id'].iloc[0]
base_c = df_cli_mes['clientes'].iloc[0]
base_p = df_cli_mes['pedidos'].iloc[0]
base_t = df_cli_mes['ticket'].iloc[0]

meses = df_cli_mes['month/year'].values
idx_s = (df_seller_mes['seller_id'] / base_s * 100).round(0).values
idx_c = (df_cli_mes['clientes'] / base_c * 100).round(0).values
idx_p = (df_cli_mes['pedidos'] / base_p * 100).round(0).values
idx_t = (df_cli_mes['ticket'] / base_t * 100).round(0).values

fig = go.Figure()
for name, vals, color in [('Clientes Únicos', idx_c, '#2980b9'), ('Sellers Ativos', idx_s, '#e67e22'),
                           ('Pedidos', idx_p, '#27ae60'), ('Ticket Médio', idx_t, '#e74c3c')]:
    fig.add_trace(go.Scatter(x=meses, y=vals, name=name, mode='lines+markers+text',
        line=dict(color=color, width=3), marker=dict(size=6),
        text=[f'{int(v)}' for v in vals], textposition='top center',
        textfont=dict(size=9, color=color)))

fig.add_hline(y=100, line=dict(color='#bdc3c7', width=1, dash='dash'))
fig.add_vline(x='11/2017', line=dict(color='#2c3e50', width=2, dash='dash'))
fig.add_annotation(x='11/2017', y=1.05, yref='paper', text='Black Friday', showarrow=False)

fig.update_layout(
    title='Sellers, Clientes, Pedidos e Ticket Médio — Crescimento Relativo (Base 100)',
    yaxis_title='Índice (Base 100)', plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center'),
    xaxis=dict(showgrid=False, tickangle=-45), yaxis=dict(showgrid=True, gridcolor='rgba(0,0,0,0.05)'),
    width=1200, height=550, margin=dict(t=100, b=80, l=60, r=40)
)
fig.write_image('grafico_base100.png', scale=3)
fig.show()

## 5. Slide 4 — A Base É Forte (Categorias + Estados)

In [7]:
df_revenue_by_category['order_purchase_timestamp'] = pd.to_datetime(df_revenue_by_category['order_purchase_timestamp'])
df_revenue_by_category['year'] = df_revenue_by_category['order_purchase_timestamp'].dt.year
df_customers_x_pay['year'] = df_customers_x_pay['order_purchase_timestamp'].dt.year

# Top 5 Categorias
top5_cats = df_revenue_by_category.groupby('product_category_name')['payment_value'].sum().nlargest(5).index.tolist()
df_cat = df_revenue_by_category[df_revenue_by_category['product_category_name'].isin(top5_cats)]
df_cat = df_cat.groupby(['year','product_category_name'])['payment_value'].sum().reset_index()

# Top 5 Estados
top5_est = df_customers_x_pay.groupby('customer_state')['payment_value'].sum().nlargest(5).index.tolist()
df_est = df_customers_x_pay[df_customers_x_pay['customer_state'].isin(top5_est)]
df_est = df_est.groupby(['year','customer_state'])['payment_value'].sum().reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Top 5 Categorias — Receita 2017 vs 2018', 'Top 5 Estados — Receita 2017 vs 2018'])

for yr, cor, show in [(2017, '#2980b9', True), (2018, '#85c1e9', True)]:
    d = df_cat[df_cat['year']==yr].sort_values('payment_value', ascending=True)
    fig.add_trace(go.Bar(y=d['product_category_name'], x=d['payment_value'], name=str(yr), orientation='h',
        marker_color=cor, text=[f'R$ {v/1e6:.1f}M' for v in d['payment_value']], textposition='inside',
        showlegend=show), row=1, col=1)

for yr, cor in [(2017, '#2980b9'), (2018, '#85c1e9')]:
    d = df_est[df_est['year']==yr].sort_values('payment_value', ascending=True)
    fig.add_trace(go.Bar(y=d['customer_state'], x=d['payment_value'], name=str(yr), orientation='h',
        marker_color=cor, text=[f'R$ {v/1e6:.1f}M' for v in d['payment_value']], textposition='inside',
        showlegend=False), row=1, col=2)

fig.update_layout(barmode='group', plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation='h', x=0.5, y=1.08, xanchor='center'),
    xaxis=dict(showgrid=False, showticklabels=False), xaxis2=dict(showgrid=False, showticklabels=False),
    width=1200, height=500, margin=dict(t=100, b=120, l=180, r=40))

# KPIs no rodapé
total_cats = df_revenue_by_category['product_category_name'].nunique()
total_est = df_customers_x_pay['customer_state'].nunique()
for x, txt in [(0.15, f'{total_cats} categorias ativas'), (0.5, f'Presente em {total_est} estados'), (0.85, '+20% no catálogo')]:
    fig.add_annotation(x=x, y=-0.18, xref='paper', yref='paper', text=f'<b>{txt}</b>', showarrow=False,
        font=dict(size=14, color='#2980b9'), bgcolor='rgba(41,128,185,0.1)', bordercolor='#2980b9', borderwidth=1, borderpad=6)

fig.write_image('grafico_base_forte.png', scale=3)
fig.show()

## 6. Slide 5 — Cohort Analysis (Onde o Balde Fura)

In [8]:
# Primeira compra de cada cliente
df_primeira = df_customers_review.groupby('customer_unique_id')['ano_mes'].min().reset_index()
df_primeira.columns = ['customer_unique_id', 'cohort']

df_cohort = df_customers_review.merge(df_primeira, on='customer_unique_id', how='left')
df_cohort['cohort_dt'] = pd.to_datetime(df_cohort['cohort'])
df_cohort['compra_dt'] = pd.to_datetime(df_cohort['ano_mes'])
df_cohort['mes_relativo'] = (df_cohort['compra_dt'].dt.to_period('M') - df_cohort['cohort_dt'].dt.to_period('M')).apply(lambda x: x.n)

# Pivot e retenção
df_pivot = df_cohort.groupby(['cohort','mes_relativo'])['customer_unique_id'].nunique().reset_index()
df_pivot = df_pivot.pivot(index='cohort', columns='mes_relativo', values='customer_unique_id')
df_retencao = df_pivot.divide(df_pivot[0], axis=0) * 100

# Filtro trimestral
cohorts = ['2017-01','2017-04','2017-07','2017-10','2018-01','2018-04']
df_retencao = df_retencao[df_retencao.index.isin(cohorts)].iloc[:, :5]

# Labels
x_labels = ['Mês 0', 'Mês 1', 'Mês 2', 'Mês 3', 'Mês 4']
y_labels = [pd.to_datetime(m).strftime('%b %Y') for m in df_retencao.index]
cohort_sizes = df_pivot[0].reindex(df_retencao.index).fillna(0).astype(int)

# Texto
custom_text = [[f'{v:.1f}%' if not np.isnan(v) else '' for v in row] for row in df_retencao.values]

fig = go.Figure(go.Heatmap(
    z=df_retencao.values, x=x_labels, y=y_labels, text=custom_text, texttemplate='%{text}',
    textfont=dict(size=14, family='Arial Black', color='#2c3e50'), showscale=False,
    colorscale=[[0,'#fff5f5'],[0.02,'#fed7d7'],[0.05,'#feb2b2'],[0.1,'#fc8181'],[0.25,'#e53e3e'],[0.5,'#c53030'],[1,'#1a1a2e']],
    xgap=5, ygap=5
))

# Base à esquerda
for cohort, size in zip(y_labels, cohort_sizes):
    fig.add_annotation(x=-0.7, y=cohort, text=f'n={size:,}', showarrow=False,
        font=dict(size=10, color='#999'), xanchor='right')

# Insight
fig.add_annotation(x=0.5, y=-0.22, xref='paper', yref='paper',
    text='A retenção despenca para menos de 1% já no primeiro mês — em todos os cohorts, sem exceção',
    showarrow=False, font=dict(size=13, color='#c53030'), bgcolor='#fff5f5', bordercolor='#e53e3e', borderwidth=1, borderpad=12)

fig.update_layout(
    title=dict(text='Cohort Analysis — Onde o Balde Fura', font=dict(size=22, color='#1a1a2e', family='Arial Black'), x=0.5),
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(tickfont=dict(size=13), showgrid=False, side='top'),
    yaxis=dict(tickfont=dict(size=12), autorange='reversed', showgrid=False),
    width=750, height=450, margin=dict(t=110, b=90, l=130, r=30)
)
fig.write_image('grafico_cohort_v2.png', scale=3)
fig.show()

## 7. Slide 5 — Receita Novos vs Recorrentes

In [9]:
df_primeira_ano = df_customers_review.groupby('customer_unique_id')['year'].min().reset_index()
df_primeira_ano.columns = ['customer_unique_id', 'ano_primeira_compra']
df_tipo = df_customers_review.merge(df_primeira_ano, on='customer_unique_id', how='left')
df_tipo['tipo_cliente'] = df_tipo.apply(lambda x: 'Novo' if x['year'] == x['ano_primeira_compra'] else 'Recorrente', axis=1)

df_receita_tipo = df_tipo.groupby(['year','tipo_cliente'])['payment_value'].sum().reset_index()
df_receita_tipo = df_receita_tipo[df_receita_tipo['year'].isin([2017, 2018])]

# Totais
total_novo = df_receita_tipo[df_receita_tipo['tipo_cliente']=='Novo']['payment_value'].sum()
total_rec = df_receita_tipo[df_receita_tipo['tipo_cliente']=='Recorrente']['payment_value'].sum()
print(f'Receita Novos: R$ {total_novo/1e6:.2f}M ({total_novo/(total_novo+total_rec)*100:.1f}%)')
print(f'Receita Recorrentes: R$ {total_rec/1e6:.2f}M ({total_rec/(total_novo+total_rec)*100:.1f}%)')

Receita Novos: R$ 15.34M (99.3%)
Receita Recorrentes: R$ 0.11M (0.7%)


## 8. Slide 6 — Balde Furado (Clientes vs Pedidos Base 100)

In [10]:
df_merged = df_orders_delivered.merge(df_customers[['customer_id','customer_unique_id']], on='customer_id', how='left')
df_merged['ano_mes'] = df_merged['order_purchase_timestamp'].dt.to_period('M').astype(str)
df_mes = df_merged.groupby('ano_mes').agg(clientes=('customer_unique_id','nunique'), pedidos=('order_id','nunique')).reset_index()
df_mes = df_mes[df_mes['ano_mes'] >= '2017-01'].sort_values('ano_mes')

base_c = df_mes['clientes'].iloc[0]
base_p = df_mes['pedidos'].iloc[0]
df_mes['idx_clientes'] = (df_mes['clientes'] / base_c * 100).round(0)
df_mes['idx_pedidos'] = (df_mes['pedidos'] / base_p * 100).round(0)

fig = go.Figure()
for name, col, color in [('Clientes Únicos', 'idx_clientes', '#2980b9'), ('Pedidos', 'idx_pedidos', '#27ae60')]:
    fig.add_trace(go.Scatter(x=df_mes['ano_mes'], y=df_mes[col], name=name, mode='lines+markers+text',
        line=dict(color=color, width=3), text=[f'{int(v)}' for v in df_mes[col]], textposition='top center',
        textfont=dict(size=10, color=color)))

fig.add_hline(y=100, line=dict(color='#bdc3c7', width=1, dash='dash'))
fig.add_vline(x='2017-11', line=dict(color='#2c3e50', width=2, dash='dash'))
fig.add_annotation(x='2017-11', y=1.05, yref='paper', text='Black Friday', showarrow=False)

fig.update_layout(
    title='Clientes e Pedidos — Crescimento Relativo (Base 100)',
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    yaxis_title='Índice (Base 100)', legend=dict(orientation='h', y=1.02, x=0.5, xanchor='center'),
    xaxis=dict(showgrid=False, tickangle=-45), yaxis=dict(showgrid=True, gridcolor='rgba(0,0,0,0.05)'),
    width=1100, height=500, margin=dict(t=100, b=80, l=60, r=40)
)
fig.write_image('grafico_clientes_pedidos_base100.png', scale=3)
fig.show()

## 9. Slide 7 — Cenários de Retenção

In [11]:
total_clientes = df_customers_review['customer_unique_id'].nunique()
ticket_medio = df_customers_review['payment_value'].sum() / df_customers_review['order_id'].nunique()

# Receita recorrente atual
rec_recorrente = df_tipo[df_tipo['tipo_cliente']=='Recorrente']['payment_value'].sum()

print(f'Total clientes: {total_clientes:,}')
print(f'Ticket médio: R$ {ticket_medio:.2f}')
print(f'Receita recorrente atual: R$ {rec_recorrente/1e6:.4f}M')
print()

# Cenários
cenarios = ['Retenção Atual\n(~3%)', 'Retenção 10%', 'Retenção 20%', 'Retenção 30%']
valores = [rec_recorrente]
for taxa in [10, 20, 30]:
    val = total_clientes * (taxa/100) * ticket_medio
    valores.append(val)
    print(f'Retenção {taxa}%: R$ {val/1e6:.1f}M')

fig = go.Figure(go.Bar(
    x=cenarios, y=valores, marker=dict(color=['#e74c3c','#85c1e9','#2980b9','#1a5276']),
    text=[f'R$ {v/1e6:.1f}M' for v in valores], textposition='outside',
    textfont=dict(size=16, color='#2c3e50', family='Arial'), width=0.6
))

fig.add_annotation(x='Retenção 20%', y=valores[2], text='<b>META</b>', showarrow=False,
    yshift=45, font=dict(size=18, color='#27ae60'))

fig.update_layout(
    title=dict(text='Custo de Não Retenção — Receita por Cenário', x=0.5),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=False), yaxis=dict(showgrid=False, showticklabels=False, range=[0, max(valores)*1.25]),
    showlegend=False, width=600, height=470, margin=dict(t=100, b=60, l=30, r=30)
)
fig.write_image('grafico_cenarios.png', scale=3)
fig.show()

Total clientes: 93,357
Ticket médio: R$ 160.58
Receita recorrente atual: R$ 0.1074M

Retenção 10%: R$ 1.5M
Retenção 20%: R$ 3.0M
Retenção 30%: R$ 4.5M


## 10. Slide 8 — Concentração de Categorias

In [12]:
df_cat_receita = df_revenue_by_category.groupby('product_category_name')['payment_value'].sum().sort_values(ascending=False).reset_index()
df_cat_receita.columns = ['categoria', 'receita']
df_cat_receita['pct'] = df_cat_receita['receita'] / df_cat_receita['receita'].sum() * 100
n_categorias = len(df_cat_receita)

top10 = df_cat_receita.head(10).copy()
outras = pd.DataFrame({'categoria': [f'Outras {n_categorias-10} categorias'], 'receita': [df_cat_receita.iloc[10:]['receita'].sum()]})
df_plot = pd.concat([top10, outras], ignore_index=True)

top5_total = df_cat_receita.head(5)['receita'].sum()
total = df_cat_receita['receita'].sum()
cores = ['#e74c3c']*5 + ['#85c1e9']*5 + ['#d5dbdb']

fig = go.Figure(go.Bar(
    x=df_plot['categoria'], y=df_plot['receita'], marker=dict(color=cores),
    text=[f'R$ {v/1e6:.1f}M' for v in df_plot['receita']], textposition='outside',
    textfont=dict(size=11, color='#2c3e50')
))

fig.add_vline(x=4.5, line=dict(color='#e74c3c', width=2, dash='dash'), opacity=0.5)
fig.add_annotation(x=0.35, y=0.95, xref='paper', yref='paper',
    text=f'<b>Top 5 categorias = {top5_total/total*100:.0f}% da receita</b>',
    showarrow=False, font=dict(size=15, color='#e74c3c'), bgcolor='#fff5f5', bordercolor='#e74c3c', borderwidth=1, borderpad=10)

fig.update_layout(
    title=dict(text='Concentração de Receita — Falta de Diversificação', x=0.5),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=False, tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(showgrid=False, showticklabels=False),
    width=800, height=450, margin=dict(t=80, b=100, l=30, r=30)
)
fig.write_image('grafico_concentracao_categorias.png', scale=3)
fig.show()

## 11. Slide 9 — Tabela Diagnóstico Final

In [13]:
categorias = [
    ('OK', 'Logística excelente', 'Entregas no prazo', '#d5f5e3'),
    ('OK', 'Reviews positivos', 'Clientes satisfeitos', '#d5f5e3'),
    ('OK', 'Catálogo crescendo', '20% mais produtos em 2018', '#d5f5e3'),
    ('ALERTA', 'Categorias com baixo volume', 'Oportunidade de curadoria', '#fef9e7'),
    ('ALERTA', 'Mix sem diversificação', '72 categorias estagnadas', '#fef9e7'),
    ('CRÍTICO', 'Retenção crítica', '+90% dos clientes compram apenas 1x', '#fadbd8'),
    ('CRÍTICO', 'Dependência de aquisição', 'Modelo insustentável sem retenção', '#fadbd8'),
]

status_colors = ['#27ae60']*3 + ['#f39c12']*2 + ['#e74c3c']*2

fig = go.Figure(go.Table(
    columnwidth=[40, 200, 350],
    header=dict(values=['<b>Status</b>','<b>Ponto</b>','<b>Diagnóstico</b>'],
        fill_color='#2c3e50', font=dict(color='white', size=16), align='center', height=45, line=dict(color='white', width=2)),
    cells=dict(
        values=[[f'<b>{c[0]}</b>' for c in categorias], [f'<b>{c[1]}</b>' for c in categorias], [c[2] for c in categorias]],
        fill_color=[[c[3] for c in categorias]]*3,
        font=dict(color=[status_colors, ['#2c3e50']*7, ['#2c3e50']*7], size=15),
        align=['center','left','left'], height=45, line=dict(color='white', width=2)
    )
))

fig.update_layout(title=dict(text='Diagnóstico Olist 2017-2018', x=0.5),
    plot_bgcolor='white', paper_bgcolor='white', width=1000, height=480, margin=dict(t=80, b=40, l=40, r=40))
fig.write_image('grafico_tabela_diagnostico.png', scale=3)
fig.show()

## 12. KPIs Resumo

In [14]:
# Resumo final
print('='*50)
print('RESUMO — OLIST 2017-2018')
print('='*50)
print(f'Receita total: R$ {df_customers_review["payment_value"].sum()/1e6:.1f}M')
print(f'Pedidos entregues: {df_orders_delivered["order_id"].nunique():,}')
print(f'Clientes únicos: {total_clientes:,}')
print(f'Ticket médio: R$ {ticket_medio:.2f}')
print(f'Taxa de recompra: {(df_tipo[df_tipo["tipo_cliente"]=="Recorrente"]["customer_unique_id"].nunique() / total_clientes * 100):.2f}%')
print(f'Receita recorrente: R$ {rec_recorrente/1e6:.4f}M ({rec_recorrente/(total_novo+total_rec)*100:.2f}%)')
print(f'Entregas no prazo: ~93%')
print(f'Avaliações 5 estrelas: ~59%')

RESUMO — OLIST 2017-2018
Receita total: R$ 15.5M
Pedidos entregues: 96,478
Clientes únicos: 93,357
Ticket médio: R$ 160.58
Taxa de recompra: 0.70%
Receita recorrente: R$ 0.1074M (0.70%)
Entregas no prazo: ~93%
Avaliações 5 estrelas: ~59%
